# Sonar Signal Classification & Dataset Optimization (UCI Sonar Dataset)
**Author:** Matheus Paixão  
**Domain:** Machine Learning Engineering, Feature Scaling & Signal Processing  
**Stack:** Python, Scikit-Learn (SVM, StandardScaler, SelectKBest, GridSearchCV), Pandas

## Overview
Optimizing a baseline Support Vector Machine (SVM) pipeline for classifying sonar signals returned from metal cylinders (Mines) vs. Rocks using the UCI Connectionist Bench dataset (60 sonar frequency channels).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [2]:
# URL da base Sonar
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/undocumented/connectionist-bench/sonar/sonar.all-data"

df = pd.read_csv(url, header=None)

In [3]:
# As 60 primeiras colunas são features, a última é a classe ('R' ou 'M')
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Codificar target (R=0, M=1)
y = y.map({'R': 0, 'M': 1})


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)


svm = SVC(random_state=42)
svm.fit(X_train, y_train)


y_pred = svm.predict(X_test)

# Avaliar resultados
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Acurácia INICIAL COM MINIMO DE INTERVENÇÃO: {acc:.4f}")
print("Matriz de Confusão:")
print(cm)

Acurácia INICIAL COM MINIMO DE INTERVENÇÃO: 0.7937
Matriz de Confusão:
[[20  9]
 [ 4 30]]


In [4]:
# AGORA VAMOS AJUSTAR

In [5]:
def create_features(df):
    df = df.copy()
    df['mean_even'] = df.iloc[:, ::2].mean(axis=1)
    df['mean_odd'] = df.iloc[:, 1::2].mean(axis=1)
    df['sum_first_10'] = df.iloc[:, :10].sum(axis=1)
    df['prod_0_1'] = df.iloc[:, 0] * df.iloc[:, 1]
    df['diff_2_3'] = df.iloc[:, 2] - df.iloc[:, 3]
    df.columns = df.columns.astype(str)
    return df


X = df.iloc[:, :-1]
y = df.iloc[:, -1].map({'R': 0, 'M': 1})

X_fe = create_features(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_fe, y, test_size=0.3, random_state=42, stratify=y)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif, k=50)),  # Aumentando para 50 features
    ('svm', SVC(class_weight='balanced', random_state=42))
])

param_grid = {
    'svm__C': [3, 4, 5, 6, 7],  # Busca mais refinada próxima a 5
    'svm__gamma': [0.05, 0.075, 0.1, 0.125, 0.15],  # Busca refinada próxima a 0.1
    'svm__kernel': ['rbf']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)


y_pred = grid_search.predict(X_test)
print(f"Acurácia FINAL APOS AJUSTES: {accuracy_score(y_test, y_pred):.4f}")
print("Matriz de Confusão:")
print(confusion_matrix(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))


Acurácia FINAL APOS AJUSTES: 0.8889
Matriz de Confusão:
[[24  5]
 [ 2 32]]

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.92      0.83      0.87        29
           1       0.86      0.94      0.90        34

    accuracy                           0.89        63
   macro avg       0.89      0.88      0.89        63
weighted avg       0.89      0.89      0.89        63

